In [77]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, cross_validate

In [78]:
train_df = pd.read_csv('../data/processed/train_cleaned.csv')
test_df = pd.read_csv('../data/processed/test_cleaned.csv')

X_train, y_train = train_df.drop(columns='target'), train_df['target']
X_test, y_test = test_df.drop(columns='target'), test_df['target']

In [79]:
numeric_cols = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_cols = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

In [80]:
dt_num_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

dt_cat_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

dt_preprocessor = ColumnTransformer([
    ("num", dt_num_proc, numeric_cols),
    ("cat", dt_cat_proc, categorical_cols)
])

dt_pipeline = Pipeline([
    ("preprocess", dt_preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

In [81]:
lr_num_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

lr_cat_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

lr_preprocessor = ColumnTransformer([
    ("num", lr_num_proc, numeric_cols),
    ("cat", lr_cat_proc, categorical_cols)
])

lr_pipeline = Pipeline([
    ("preprocess", lr_preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [82]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

models = {
    "Decision Tree": dt_pipeline,
    "Logistic Regression": lr_pipeline
}

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "F1 Std": scores["test_f1"].std()
    })

baseline_results = pd.DataFrame(results)
baseline_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,F1 Std
0,Decision Tree,0.706122,0.680475,0.702372,0.688863,0.705887,0.050057
1,Logistic Regression,0.847109,0.876753,0.783399,0.824501,0.902493,0.006867
